In [1]:
import pandas as pd
import numpy as np

##  Load full data and concatenate

In [2]:
web_1 = pd.read_csv("df_final_web_data_pt_1.csv")
web_2 = pd.read_csv("df_final_web_data_pt_2.csv")

df_web = pd.concat([web_1, web_2], ignore_index=True)

## Quality checks

In [3]:
df_web.shape

(755405, 5)

In [4]:
df_web.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04


In [5]:
df_web.info()

<class 'pandas.DataFrame'>
RangeIndex: 755405 entries, 0 to 755404
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   client_id     755405 non-null  int64
 1   visitor_id    755405 non-null  str  
 2   visit_id      755405 non-null  str  
 3   process_step  755405 non-null  str  
 4   date_time     755405 non-null  str  
dtypes: int64(1), str(4)
memory usage: 81.6 MB


In [6]:
df_web["client_id"].nunique()

120157

In [7]:
df_web.groupby("client_id").size().describe()

count    120157.000000
mean          6.286816
std           3.986973
min           1.000000
25%           5.000000
50%           5.000000
75%           7.000000
max         111.000000
dtype: float64

In [8]:
df_web["process_step"].value_counts()

process_step
start      243945
step_1     163193
step_2     133062
step_3     112242
confirm    102963
Name: count, dtype: int64

In [9]:
df_web.duplicated().sum()

np.int64(10764)

In [10]:
# Convert to datetime so we can compute durations and sort chronologically
df_web["date_time"] = pd.to_datetime(df_web["date_time"])
print(df_web["date_time"].dtype)

datetime64[us]


In [11]:
df_web["process_step"].value_counts()

process_step
start      243945
step_1     163193
step_2     133062
step_3     112242
confirm    102963
Name: count, dtype: int64

In [12]:
df_web["process_step"].unique()

<ArrowStringArray>
['step_3', 'step_2', 'step_1', 'start', 'confirm']
Length: 5, dtype: str

In [13]:
df_web["date_time"].isna().sum()

np.int64(0)

## Sort

Sort by client → visit → time so that subsequent `shift()` operations correctly reference the previous event *within the same visit*.

In [14]:
df_web = df_web.sort_values(["client_id", "visit_id", "date_time"])

In [15]:
# Map step names to numeric positions so we can compare them arithmetically
df_web["step_num"] = df_web["process_step"].map({"start": 0, "step_1": 1, "step_2": 2, "step_3": 3, "confirm": 4})

In [16]:
# For each event, get the step number of the previous event within the same visit
df_web['previous_step_num'] = df_web.groupby('visit_id')['step_num'].shift(1)

In [17]:
# NaN previous_step_num = first event of a visit
df_web["previous_step_num"].isna().sum()

np.int64(158095)

In [18]:
df_web["visit_id"].nunique()

158095

In [19]:
# Classify each event by direction relative to the previous event
# "first_visit" applies when there's no previous step (i.e., NaN)
conditions = [
    df_web["previous_step_num"] > df_web["step_num"],
    df_web["previous_step_num"] < df_web["step_num"],
    df_web["previous_step_num"] == df_web["step_num"]
]

labels = ["backward", "forward", "repeat"]

df_web["step_direction"] = np.select(conditions, labels, default="first_visit")

In [20]:
# Seconds between this event and the previous one in the same visit
df_web["time_on_each_step"] = (df_web["date_time"] - df_web.groupby("visit_id")["date_time"].shift(1)).dt.total_seconds()

In [25]:
df_web = df_web.sort_values(['visit_id', 'date_time']).reset_index(drop=True)
df_web['next_step_num'] = df_web.groupby('visit_id')['step_num'].shift(-1)
df_web['jump_size'] = df_web['next_step_num'] - df_web['step_num']

In [26]:
time_per_step = df_web[df_web["process_step"] != "confirm"].groupby('process_step')['time_on_each_step'].mean().reset_index()

In [28]:
time_per_step.sort_values(by="time_on_each_step", ascending=False)

,process_step,time_on_each_step
0,start,139.024150
3,step_3,99.382272
2,step_2,45.788994
1,step_1,39.439692


In [31]:
df_web

,client_id,visitor_id,visit_id,process_step,date_time,step_num,previous_step_num,step_direction,time_on_each_step,next_step_num,jump_size
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,4,NaN,first_visit,NaN,4.0,0.0
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,4,4.0,repeat,52.0,NaN,NaN
2,9056452,306992881_89423906595,1000165_4190026492_760066,start,2017-06-04 01:07:29,0,NaN,first_visit,NaN,1.0,1.0
3,9056452,306992881_89423906595,1000165_4190026492_760066,step_1,2017-06-04 01:07:32,1,0.0,forward,3.0,2.0,1.0
4,9056452,306992881_89423906595,1000165_4190026492_760066,step_2,2017-06-04 01:07:56,2,1.0,forward,24.0,3.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
755400,7149380,483112224_46340533900,999992932_41666455053_671149,step_1,2017-06-06 15:46:24,1,0.0,forward,21.0,2.0,1.0
755401,7149380,483112224_46340533900,999992932_41666455053_671149,step_2,2017-06-06 15:47:32,2,1.0,forward,68.0,3.0,1.0
755402,7149380,483112224_46340533900,999992932_41666455053_671149,step_3,2017-06-06 16:01:46,3,2.0,forward,854.0,4.0,1.0
755403,7149380,483112224_46340533900,999992932_41666455053_671149,confirm,2017-06-06 16:04:08,4,3.0,forward,142.0,4.0,0.0


## Initial Insights from Web Data

### 1. Sessions per Client
- Clients have multiple sessions on average.
- There is noticeable variability: some clients interact only once, while others return multiple times.
- This suggests different levels of engagement across users.

### 2. Funnel / Drop-off Behavior
- There is a clear drop-off at each step of the process:
  - Start → Step 1 → Step 2 → Step 3 → Confirm
- A significant portion of users does not reach the final confirmation step.
- This indicates potential friction points in the process that may need further investigation.

### 3. User Journey Patterns
- Most sessions follow the expected sequence (start → step progression → confirm).
- However, some irregular patterns exist (e.g. repeated steps or incomplete journeys).
- This suggests that not all users follow a clean linear path.

### 4. Data Structure Insight
- Raw web data is event-level (multiple rows per client).
- After aggregation, we successfully transformed it into a client-level dataset.
- Each client now has a clear behavioral profile (whether they reached each step).

---

## Note
These insights are based only on web interaction data.
They should be validated and extended after merging with experiment and client datasets.

In [34]:
## Save
df_web.to_csv("events_kpi_table.csv", index=False)